Test model

In [1]:
!pip install -U transformers accelerate bitsandbytes

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.7/10.7 MB 120.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 383.7/383.7 kB 38.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 13.7 MB/s eta 0:00:00
  Attempting uninstall: accelerate
    Found existing installation: accelerate 1.12.0
    Uninstalling accelerate-1.12.0:
      Successfully uninstalled accelerate-1.12.0
  Attempting uninstall: transformers
    Found existing installation: transformers 5.0.0
    Uninstalling transformers-5.0.0:
      Successfully uninstalled transformers-5.0.0


In [4]:
# เพิ่มบรรทัดนี้เพื่อเช็คว่าตัวแปร model มีอยู่แล้วหรือไม่
if 'model' not in globals():
    print("🚀 กำลังโหลดโมเดลครั้งแรก...")

    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModelForCausalLM.from_pretrained(
        model_name,
        quantization_config=quantization_config,
        device_map="auto"
    )
    print("✅ โหลดสำเร็จ!")
else:
    print("♻️ โมเดลโหลดอยู่ในแรมแล้ว ข้ามขั้นตอนการโหลด weights")

♻️ โมเดลโหลดอยู่ในแรมแล้ว ข้ามขั้นตอนการโหลด weights


In [8]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
import torch

model_name = "Qwen/Qwen2.5-3B-Instruct"

# 1. การตั้งค่าบีบอัด (ใช้ VRAM ประมาณ 2.5GB - 3GB เท่านั้น)
quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_quant_type="nf4"
)

# 2. โหลด Tokenizer และ Model
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=quantization_config,
    device_map="auto"
)

# 3. ฟังก์ชันสำหรับแชท (รองรับภาษาไทยดีมาก)
def chat_with_qwen(prompt):
    messages = [
        {"role": "system", "content": "You are a helpful assistant. Please respond in Thai."},
        {"role": "user", "content": prompt}
    ]
    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )
    model_inputs = tokenizer([text], return_tensors="pt").to(model.device)

    generated_ids = model.generate(
        **model_inputs,
        max_new_tokens=512,
        do_sample=True,
        temperature=0.7
    )

    # ตัดส่วนที่เป็น Prompt ออก ให้เหลือแค่คำตอบ
    generated_ids = [
        output_ids[len(input_ids):] for input_ids, output_ids in zip(model_inputs.input_ids, generated_ids)
    ]

    response = tokenizer.batch_decode(generated_ids, skip_special_tokens=True)[0]
    return response

# ทดสอบใช้งาน
print(chat_with_qwen("คุณสามามารถแต่งเพลง สักท่อนให้ฉันได้ไหม แนวสติง"))

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

แน่นอนครับ นี่คือท่อนเพลงสติงแบบสกอร์สกรีมที่คุณอาจจะชอบ:

(ทำนองแบบสติง)
D D C G
C G F D
G F E C
E C Bb G

เนื้อร้อง:
"Whispering wind, it's time to sing,
In the silence of the night, we find our thing.
Melodies so sweet, so pure and true,
In the heart of the night, let's take a view."

ขออภัยว่าทำนองอาจไม่สมบูรณ์เท่าที่ควรครับ เพราะการแต่งทำนองสติงมักจะใช้ดนตรีพื้นฐานและเทคนิคการเขียนทำนองที่ซับซ้อนกว่านี้ แต่หวังว่าท่อนนี้จะช่วยให้คุณรู้สึกว่ากำลังแต่งเพลงสติงแล้วครับ.


### end Test model

In [ ]:
# ติดตั้ง Library ที่จำเป็น
!pip install -U transformers accelerate bitsandbytes

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
import torch

model_name = "Qwen/Qwen2.5-3B-Instruct"

# 1. เช็คว่าโหลดไปหรือยัง ถ้ายังไม่โหลดค่อยทำ
if 'model' not in globals():
    print("🚀 กำลังเตรียมโหลด Qwen2.5-3B เข้าแรม GPU...")

    quantization_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_compute_dtype=torch.bfloat16,
        bnb_4bit_quant_type="nf4"
    )

    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModelForCausalLM.from_pretrained(
        model_name,
        quantization_config=quantization_config,
        device_map="auto"
    )

    # เก็บประวัติการคุย (History) ไว้ในลิสต์นี้
    chat_history = [
        {"role": "system", "content": "You are a helpful assistant. Please respond in Thai."}
    ]
    print("✅ โหลดสำเร็จ! พร้อมใช้งานแล้วครับ")
else:
    print("♻️ โมเดลโหลดรอไว้อยู่แล้ว พร้อมรันต่อได้เลย!")

### จดจำ Chat เดิม

In [ ]:
# --- ใส่คำถามของคุณตรงนี้ ---
user_input = "แล้วกุ้งแม่น้ำล่ะ ทำต้มยำเหมือนกันไหม?"

# 1. เพิ่มคำถามใหม่ลงในประวัติ
chat_history.append({"role": "user", "content": user_input})

# 2. เตรียม Prompt จากประวัติทั้งหมด
text = tokenizer.apply_chat_template(
    chat_history,
    tokenize=False,
    add_generation_prompt=True
)
model_inputs = tokenizer([text], return_tensors="pt").to(model.device)

# 3. ให้ AI สร้างคำตอบ
generated_ids = model.generate(
    **model_inputs,
    max_new_tokens=512,
    do_sample=True,
    temperature=0.7
)

# 4. ตัดส่วนคำถามออกให้เหลือแค่คำตอบใหม่
generated_ids = [
    output_ids[len(input_ids):] for input_ids, output_ids in zip(model_inputs.input_ids, generated_ids)
]
response = tokenizer.batch_decode(generated_ids, skip_special_tokens=True)[0]

# 5. บันทึกคำตอบของ AI ลงในประวัติ เพื่อใช้คุยต่อในรอบหน้า
chat_history.append({"role": "assistant", "content": response})

# --- แสดงผล ---
print(f"👤 คุณ: {user_input}")
print(f"🤖 AI: {response}")
print(f"\n(ประวัติการคุยตอนนี้มีทั้งหมด {len(chat_history)} ข้อความ)")

ถ้าอยากเริ่มคุยเรื่องใหม่ (ล้างสมอง): ให้กลับไปรัน Cell 2 อีกรอบ (หรือเพิ่มบรรทัด chat_history = [{"role": "system", "content": "..."}] ใน Cell 3 แล้วรันหนึ่งครั้ง) เพื่อล้างประวัติเก่าครับ

ถ้า AI ตอบสั้นไป: ปรับค่า max_new_tokens ใน Cell 3 ให้มากขึ้น (เช่น 1024)

ถ้าอยากให้ AI มีบุคลิกเปลี่ยนไป: แก้ไขข้อความใน system prompt ที่อยู่ใน Cell 2 ครับ